In [0]:

from pyspark.sql.functions import current_timestamp, col

# Source file path in Databricks Unity Catalog Volume
source_path = "/Volumes/etlpipelines/default/sana/Global-Superstore - Copy.csv"

# Read raw CSV data
df_bronze = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("encoding", "ISO-8859-1")
    .option("mode", "PERMISSIVE")
    .load(source_path)
)

# Sanitize column names for Delta compatibility (replace invalid characters with underscore)
import re
sanitized_cols = [re.sub(r"[ ,;{}()\n\t=]", "_", c) for c in df_bronze.columns]
df_bronze = df_bronze.toDF(*sanitized_cols)


# Add ingestion metadata
df_bronze = (
    df_bronze
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

# Save as Bronze Delta table
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("etlpipelines.default.bronze_global_superstore")
)

# Validate Bronze data
display(df_bronze)
print("Bronze row count:", df_bronze.count())


Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,City,State,Country,Postal_Code,Market,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit,Shipping_Cost,Order_Priority,_ingestion_timestamp,_source_file
32298,CA-2012-124891,7/31/2012,7/31/2012,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,United States,10024,US,East,TEC-AC-10003033,Technology,Accessories,Plantronics CS510 - Over-the-Head monaural Wireless Headset System,2309.65,7,0,762.1845,933.57,Critical,2026-10-17T12:40:14.272Z,dbfs:/Volumes/etlpipelines/default/sana/Global-Superstore%20-%20Copy.csv
26341,IN-2013-77878,2/5/2013,2/7/2013,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,Australia,null,APAC,Oceania,FUR-CH-10003950,Furniture,Chairs,"Novimex Executive Leather Armchair, Black",3709.395,9,0.1,-288.765,923.63,Critical,2026-10-17T12:40:14.272Z,dbfs:/Volumes/etlpipelines/default/sana/Global-Superstore%20-%20Copy.csv
25330,IN-2013-71249,10/17/2013,10/18/2013,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,Australia,null,APAC,Oceania,TEC-PH-10004664,Technology,Phones,"Nokia Smart Phone, with Caller ID",5175.171,9,0.1,919.971,915.49,Medium,2026-10-17T12:40:14.272Z,dbfs:/Volumes/etlpipelines/default/sana/Global-Superstore%20-%20Copy.csv
13524,ES-2013-1579342,1/28/2013,1/30/2013,First Class,KM-16375,Katherine Murray,Home Office,Berlin,Berlin,Germany,null,EU,Central,TEC-PH-10004583,Technology,Phones,"Motorola Smart Phone, Cordless",2892.51,5,0.1,-96.54,910.16,Medium,2026-10-17T12:40:14.272Z,dbfs:/Volumes/etlpipelines/default/sana/Global-Superstore%20-%20Copy.csv
47221,SG-2013-4320,11/5/2013,11/6/2013,Same Day,RH-9495,Rick Hansen,Consumer,Dakar,Dakar,Senegal,null,Africa,Africa,TEC-SHA-10000501,Technology,Copiers,"Sharp Wireless Fax, High-Speed",2832.96,8,0,311.52,903.04,Critical,2026-10-17T12:40:14.272Z,dbfs:/Volumes/etlpipelines/default/sana/Global-Superstore%20-%20Copy.csv
22732,IN-2013-42360,6/28/2013,7/1/2013,Second Class,JM-15655,Jim Mitchum,Corporate,Sydney,New South Wales,Australia,null,APAC,Oceania,TEC-PH-10000030,Technology,Phones,"Samsung Smart Phone, with Caller ID",2862.675,5,0.1,763.275,897.35,Critical,2026-10-17T12:40:14.272Z,dbfs:/Volumes/etlpipelines/default/sana/Global-Superstore%20-%20Copy.csv
30570,IN-2011-81826,11/7/2011,11/9/2011,First Class,TS-21340,Toby Swindell,Consumer,Porirua,Wellington,New Zealand,null,APAC,Oceania,FUR-CH-10004050,Furniture,Chairs,"Novimex Executive Leather Armchair, Adjustable",1822.08,4,0,564.84,894.77,Critical,2026-10-17T12:40:14.272Z,dbfs:/Volumes/etlpipelines/default/sana/Global-Superstore%20-%20Copy.csv
31192,IN-2012-86369,4/14/2012,4/18/2012,Standard Class,MB-18085,Mick Brown,Consumer,Hamilton,Waikato,New Zealand,null,APAC,Oceania,FUR-TA-10002958,Furniture,Tables,"Chromcraft Conference Table, Fully Assembled",5244.84,6,0,996.48,878.38,High,2026-10-17T12:40:14.272Z,dbfs:/Volumes/etlpipelines/default/sana/Global-Superstore%20-%20Copy.csv
40155,CA-2014-135909,10/14/2014,10/21/2014,Standard Class,JW-15220,Jane Waco,Corporate,Sacramento,California,United States,95823,US,West,OFF-BI-10003527,Office Supplies,Binders,Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,5083.96,5,0.2,1906.485,867.69,Low,2026-10-17T12:40:14.272Z,dbfs:/Volumes/etlpipelines/default/sana/Global-Superstore%20-%20Copy.csv
40936,CA-2012-116638,1/28/2012,1/31/2012,Second Class,JH-15985,Joseph Holt,Consumer,Concord,North Carolina,United States,28027,US,South,FUR-TA-10000198,Furniture,Tables,Chromcraft Bull-Nose Wood Oval Conference Tables & Bases,4297.644,13,0.4,-1862.3124,865.74,Critical,2026-10-17T12:40:14.272Z,dbfs:/Volumes/etlpipelines/default/sana/Global-Superstore%20-%20Copy.csv


Bronze row count: 51290
